# FoodHub NYC — Revenue & Demand Intelligence

A business-focused exploratory data analysis of 1,898 food delivery orders from FoodHub, a New York-based multi-restaurant aggregator app.

## Act 1: Business Context & Analytical Framework

FoodHub connects customers to 178 restaurants across New York City through a single smartphone app. When a customer places an order, a delivery person is dispatched from FoodHub's fleet to collect and deliver the food. The customer can rate the experience afterward. FoodHub's revenue model is margin-based:

- **25% margin** on orders above $20
- **15% margin** on orders between $5 and $20

The Data Science team needs to understand demand patterns, operational performance, and customer satisfaction to support revenue optimization and retention strategy. This analysis addresses five core business questions:

1. **Demand:** Which restaurants and cuisines drive the most orders, and when?
2. **Performance:** How does delivery speed vary across the week, and where are the SLA risks?
3. **Revenue:** How is FoodHub's revenue distributed across order segments?
4. **Satisfaction:** What drives customer ratings — and why are 39% of orders unrated?
5. **Opportunity:** Which restaurants merit promotional investment, and which customers merit retention incentives?

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

## Act 2: Data Audit & Quality Assessment

*Before analysing anything, we verify data integrity: shape, types, nulls, and the structural quirks that will affect downstream analysis.*

In [ ]:
df = pd.read_csv('../data/foodhub_order.csv')
df.head()

In [ ]:
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Unique orders:    {df["order_id"].nunique():,}')
print(f'Unique customers: {df["customer_id"].nunique():,}')
print(f'Unique restaurants: {df["restaurant_name"].nunique():,}')
print(f'Cuisine categories: {df["cuisine_type"].nunique():,}')

In [ ]:
df.info()

In [ ]:
null_check = df.isnull().sum()
print('Missing values per column:')
print(null_check)
print(f'\nTotal missing values: {null_check.sum()}')

In [ ]:
df.describe().round(2)

In [ ]:
df['total_time'] = df['food_preparation_time'] + df['delivery_time']
print('Derived column total_time = food_preparation_time + delivery_time')
df[['food_preparation_time', 'delivery_time', 'total_time']].describe().round(2)

In [ ]:
rating_counts = df['rating'].value_counts()
unrated = df[df['rating'] == 'Not given'].shape[0]
unrated_pct = round(unrated / len(df) * 100, 2)
print(f'Rating distribution:\n{rating_counts}')
print(f'\nUnrated orders: {unrated:,} ({unrated_pct}%)')

### Data Audit Findings

The dataset is clean: **no missing values** across all 9 columns and 1,898 records. Two structural notes worth flagging before analysis:

- The `rating` column is stored as **object type** containing the strings `"3"`, `"4"`, `"5"`, or `"Not given"`. Type conversion to integer is required before any numeric rating analysis, and `"Not given"` rows must be filtered out first.
- **736 orders (38.77%) have no customer rating.** This is not a data quality issue — it is a product gap. Almost 4 in 10 customers complete a delivery without submitting feedback, creating a structural blind spot in satisfaction measurement.

## Act 3: Demand & Volume Patterns

*What are customers ordering, when are they ordering, and how much are they spending?*

In [ ]:
cuisine_order = df['cuisine_type'].value_counts().index

fig, ax = plt.subplots(figsize=(14, 6))
bars = sns.countplot(data=df, x='cuisine_type', order=cuisine_order,
                     hue='cuisine_type', palette='Blues_d', legend=False, ax=ax)
ax.set_title('Order Volume by Cuisine Type', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Cuisine Type', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.bar_label(bars.containers[0], fmt='%d', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/cuisine_order_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_cost = df['cost_of_the_order'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df, x='cost_of_the_order', bins=30,
             color='#E8603C', alpha=0.85, ax=ax)
ax.axvline(20, color='#1a1a1a', linestyle='--', linewidth=1.8,
           label='$20 margin threshold (25% → 15%)')
ax.axvline(mean_cost, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Mean ${mean_cost:.2f}')
ax.set_title('Distribution of Order Cost', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Order Cost (USD)', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/cost_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Orders above $20:  {(df['cost_of_the_order'] > 20).sum():,} ({(df['cost_of_the_order'] > 20).mean()*100:.2f}%)")
print(f"Orders $5–$20:     {((df['cost_of_the_order'] > 5) & (df['cost_of_the_order'] <= 20)).sum():,}")
print(f"Median order cost: ${df['cost_of_the_order'].median():.2f}")

In [ ]:
day_counts = df['day_of_the_week'].value_counts()
ratio = day_counts['Weekend'] / day_counts['Weekday']

fig, ax = plt.subplots(figsize=(8, 5))
bars = sns.countplot(data=df, x='day_of_the_week',
                     order=['Weekend', 'Weekday'],
                     hue='day_of_the_week',
                     palette={'Weekend': '#E8603C', 'Weekday': '#7eb3d4'},
                     legend=False, ax=ax)
ax.set_title('Order Volume: Weekend vs. Weekday', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Day Type', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.bar_label(bars.containers[0], fmt='%d', padding=3, fontsize=12)
ax.text(0.97, 0.93, f'Weekend : Weekday = {ratio:.1f}×',
        transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='#fff3ee', edgecolor='#E8603C'))
plt.tight_layout()
plt.savefig('../reports/figures/orders_by_day.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rating_order = ['Not given', '3', '4', '5']

fig, ax = plt.subplots(figsize=(9, 5))
bars = sns.countplot(data=df, x='rating', order=rating_order, hue='rating',
                     palette={'Not given': '#cccccc', '3': '#f4c47a', '4': '#f4a636', '5': '#E8603C'},
                     legend=False, ax=ax)
ax.set_title('Customer Rating Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Rating', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.bar_label(bars.containers[0], fmt='%d', padding=3, fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_prep = df['food_preparation_time'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df, x='food_preparation_time', bins=16,
             color='#E8603C', alpha=0.85, ax=ax)
ax.axvline(mean_prep, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Mean {mean_prep:.1f} min')
ax.set_title('Distribution of Food Preparation Time', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Food Preparation Time (minutes)', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/prep_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_del = df['delivery_time'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df, x='delivery_time', bins=20,
             color='#E8603C', alpha=0.85, ax=ax)
ax.axvline(mean_del, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Mean {mean_del:.1f} min')
ax.set_title('Distribution of Delivery Time', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Delivery Time (minutes)', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/delivery_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Demand Patterns — Key Observations

- **American cuisine dominates** with 584 orders (30.8%), followed by Japanese (470) and Italian (298). The top 3 cuisines account for 71% of all orders — a significant concentration on the demand side.
- **Order cost is right-skewed**: median $14.14, mean $16.50. The $20 margin threshold sits near the 72nd percentile, meaning only the top 28% of orders trigger the higher 25% margin rate.
- **Weekend volume is 2.5× weekday** (1,351 vs 547 orders). This is the single most significant operational pattern in the dataset — the platform is effectively a weekend-first business.
- **Food preparation time is bounded** between 20–35 minutes with a near-uniform distribution, suggesting restaurants operate under informal or contractual SLA constraints in this window.
- **Delivery time is right-skewed**: a tail of slow deliveries pulls the mean (24.2 min) above the median (25 min). These outliers are the primary driver of the 10.54% SLA breach rate examined in Act 4.

## Act 4: Performance & Concentration Analysis

*Which restaurants dominate demand, how does delivery performance vary by day type, and where are the SLA risks?*

In [ ]:
top_restaurants = df['restaurant_name'].value_counts().head(10)
top_rest_df = top_restaurants.reset_index()
top_rest_df.columns = ['restaurant', 'orders']

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=top_rest_df, x='orders', y='restaurant',
            hue='restaurant', palette='Blues_d', legend=False, ax=ax)
for i, v in enumerate(top_rest_df['orders']):
    ax.text(v + 1, i, str(v), va='center', fontsize=10)
ax.set_title('Top 10 Restaurants by Order Volume', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Orders', fontsize=12)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../reports/figures/top_restaurants.png', dpi=150, bbox_inches='tight')
plt.show()

top5_share = top_restaurants.head(5).sum() / len(df) * 100
print(f'Top 5 restaurants: {top5_share:.1f}% of all orders')

In [ ]:
weekend_df = df[df['day_of_the_week'] == 'Weekend']
weekend_cuisine = weekend_df['cuisine_type'].value_counts().head(8)
wc_df = weekend_cuisine.reset_index()
wc_df.columns = ['cuisine_type', 'orders']

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=wc_df, x='orders', y='cuisine_type',
            hue='cuisine_type', palette='Blues_d', legend=False, ax=ax)
for i, v in enumerate(wc_df['orders']):
    pct = v / len(weekend_df) * 100
    ax.text(v + 2, i, f'{v} ({pct:.1f}%)', va='center', fontsize=10)
ax.set_title('Top Cuisines on Weekends', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Weekend Orders', fontsize=12)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../reports/figures/weekend_cuisine.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mean_wd = df[df['day_of_the_week'] == 'Weekday']['delivery_time'].mean()
mean_we = df[df['day_of_the_week'] == 'Weekend']['delivery_time'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x='day_of_the_week', y='delivery_time',
            order=['Weekday', 'Weekend'], hue='day_of_the_week',
            palette={'Weekday': '#7eb3d4', 'Weekend': '#E8603C'},
            legend=False, ax=ax)
ax.text(0, mean_wd + 0.6, f'Mean: {mean_wd:.1f} min', ha='center', fontsize=10, color='#444')
ax.text(1, mean_we + 0.6, f'Mean: {mean_we:.1f} min', ha='center', fontsize=10, color='#444')
ax.set_title('Delivery Time: Weekday vs. Weekend', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Day Type', fontsize=12)
ax.set_ylabel('Delivery Time (minutes)', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/delivery_weekday_weekend.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Weekday mean delivery: {mean_wd:.2f} min')
print(f'Weekend mean delivery: {mean_we:.2f} min')
print(f'Delta: {mean_wd - mean_we:.2f} min slower on weekdays')

In [ ]:
mean_total = df['total_time'].mean()
over_60 = df[df['total_time'] > 60]

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(data=df, x='total_time', bins=25,
             color='#E8603C', alpha=0.75, ax=ax)
ax.axvline(60, color='#1a1a1a', linestyle='--', linewidth=2.0,
           label='60-min SLA threshold')
ax.axvline(mean_total, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Mean {mean_total:.1f} min')
ax.set_title('End-to-End Fulfillment Time (Prep + Delivery)',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Total Time (minutes)', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/total_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Orders exceeding 60 min: {len(over_60):,} ({len(over_60)/len(df)*100:.2f}%)')

In [ ]:
top_customers = df['customer_id'].value_counts().head(3).reset_index()
top_customers.columns = ['customer_id', 'order_count']
print('Top 3 customers by order frequency (discount voucher candidates):')
print(top_customers.to_string(index=False))

### Performance & Concentration Findings

- **Top 5 restaurants account for ~33% of all orders** — a supply-side concentration risk. Shake Shack alone represents ~11.5% of platform volume; a single partnership disruption there would be immediately material.
- **American cuisine leads on weekends** with 415 orders (30.7% of weekend volume), followed by Japanese. Weekend demand is effectively an American-first market.
- **Weekday delivery is 5.87 minutes slower** than weekend (28.34 vs 22.47 min) despite carrying less than half the order volume. This counter-intuitive result suggests a courier staffing or routing gap on weekdays, not a demand problem.
- **10.54% of orders (200 total) breach the 60-minute end-to-end threshold.** These are the orders most likely to generate low ratings. A proactive alert at 45 minutes would allow intervention — a revised ETA or voucher offer — before the experience fails.
- **Customer 52832 placed 13 orders**, the highest in the dataset. This cohort of repeat customers has high lifetime value and is worth protecting with retention incentives before a competitor acquires them.

## Act 5: Revenue & Satisfaction Intelligence

*How does FoodHub's revenue break down across order segments, which restaurants merit promotional investment, and what actually drives customer ratings?*

In [ ]:
orders_above_20 = df[df['cost_of_the_order'] > 20]['cost_of_the_order'].sum()
orders_5_to_20 = df[(df['cost_of_the_order'] > 5) & (df['cost_of_the_order'] <= 20)]['cost_of_the_order'].sum()

revenue_above_20 = orders_above_20 * 0.25
revenue_5_to_20 = orders_5_to_20 * 0.15
total_revenue = revenue_above_20 + revenue_5_to_20

n_above_20 = (df['cost_of_the_order'] > 20).sum()

print(f'Revenue from orders >$20  (25% margin):  ${revenue_above_20:,.2f}')
print(f'Revenue from orders $5–20 (15% margin):  ${revenue_5_to_20:,.2f}')
print(f'Total net revenue:                        ${total_revenue:,.2f}')
print()
print(f'Orders >$20:  {n_above_20:,} ({n_above_20/len(df)*100:.1f}% of orders) → {revenue_above_20/total_revenue*100:.1f}% of revenue')

In [ ]:
rated_df = df[df['rating'] != 'Not given'].copy()
rated_df['rating'] = rated_df['rating'].astype(int)

agg = rated_df.groupby('restaurant_name')['rating'].agg(['count', 'mean']).reset_index()
agg.columns = ['restaurant_name', 'rating_count', 'avg_rating']
eligible = agg[(agg['rating_count'] > 50) & (agg['avg_rating'] > 4)].sort_values('rating_count', ascending=False)

print('Restaurants qualifying for promotional offer')
print('(rating count > 50 AND avg rating > 4):')
print(eligible[['restaurant_name', 'rating_count', 'avg_rating']].round(2).to_string(index=False))

In [ ]:
cuisine_cost_order = df.groupby('cuisine_type')['cost_of_the_order'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(15, 5))
sns.boxplot(data=df, x='cuisine_type', y='cost_of_the_order',
            order=cuisine_cost_order, hue='cuisine_type',
            palette='Blues_d', legend=False, ax=ax)
ax.set_title('Order Cost by Cuisine Type', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Cuisine Type', fontsize=12)
ax.set_ylabel('Order Cost (USD)', fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/cuisine_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rating_order_num = ['3', '4', '5', 'Not given']
palette_map = {'3': '#f4c47a', '4': '#f4a636', '5': '#E8603C', 'Not given': '#cccccc'}

g = sns.catplot(data=df, x='rating', y='cost_of_the_order',
                order=rating_order_num, hue='rating', hue_order=rating_order_num,
                palette=palette_map, kind='point', height=5, aspect=1.8, legend=False)
g.set_axis_labels('Customer Rating', 'Order Cost (USD)')
g.figure.suptitle('Average Order Cost by Rating', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
g.figure.savefig('../reports/figures/rating_vs_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
g = sns.catplot(data=df, x='rating', y='delivery_time',
                order=rating_order_num, hue='rating', hue_order=rating_order_num,
                palette=palette_map, kind='point', height=5, aspect=1.8, legend=False)
g.set_axis_labels('Customer Rating', 'Delivery Time (minutes)')
g.figure.suptitle('Mean Delivery Time by Rating', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
g.figure.savefig('../reports/figures/rating_vs_delivery.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
col_list = ['cost_of_the_order', 'food_preparation_time', 'delivery_time']
corr_df = df[col_list].corr()
corr_df.columns = ['Cost', 'Prep Time', 'Delivery Time']
corr_df.index = ['Cost', 'Prep Time', 'Delivery Time']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df, annot=True, vmin=-1, vmax=1, fmt='.2f',
            cmap='Blues', linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 13})
ax.set_title('Correlation Matrix — Cost, Prep Time, Delivery Time',
             fontsize=13, fontweight='bold', pad=12)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Revenue & Satisfaction Findings

- **Net revenue: $6,166.30.** Orders above $20 (29% of volume) contribute $3,688.73 — **59.8% of total revenue**. This asymmetry makes the high-value order segment disproportionately important to protect and grow.
- **Only 4 restaurants qualify for promotional offers**: Shake Shack, The Meatball Shop, Blue Ribbon Sushi, and Blue Ribbon Fried Chicken. These are the same 4 that lead order volume — quality and popularity are co-located at the top of FoodHub's restaurant portfolio.
- **Higher order cost → better rating**: customers spending more tend to choose higher-quality restaurants, and those restaurants deliver a better experience end-to-end.
- **Longer delivery time → worse rating**: this is the clearest and most actionable satisfaction driver. Customers are far more tolerant of kitchen preparation time than of delivery lag.
- **No meaningful correlation** between cost, prep time, and delivery time (highest r = 0.09 between cost and prep time). These variables are essentially independent — meaning delivery speed can be optimized without trade-offs against order value or kitchen performance.

## Act 6: Strategic Recommendations

Based on analysis of 1,898 orders across 178 restaurants, five recommendations are prioritized by potential business impact.

---

### P0 — Close the Rating Gap

**38.77% unrated orders is the highest-priority finding.** Without post-delivery feedback on 4 in 10 orders, FoodHub cannot enforce restaurant quality standards, detect early churn signals, or build a reliable satisfaction index for data-driven partner decisions.

**Action:** Implement a one-tap rating prompt (push notification or in-app modal) triggered 5 minutes after confirmed delivery. Target: reduce unrated rate from 39% to below 15% within 90 days. Measure: weekly rating completion rate by restaurant segment.

---

### P1 — Activate Weekday Demand

Weekend orders are 2.5× weekday orders, yet nothing in the data suggests weekday-specific promotions are in place. Closing even half the gap would increase weekly volume by approximately 40%.

**Action:** Run time-limited weekday discounts on American and Japanese cuisine (dominant weekend categories) with targeting by geography and order history. Measure incremental volume against a holdout control group to isolate promotional lift.

---

### P2 — Diversify the Restaurant Portfolio

The top 5 restaurants handle 33% of all orders. A single operational disruption — closure, pricing dispute, or partnership end — at Shake Shack would immediately affect ~11.5% of platform volume. This is a material supply-side risk.

**Action:** Recruit 10–15 high-quality restaurants in underrepresented categories (Mexican, Indian, Thai) with incentive structures that accelerate early orders. Set a diversity target: no single restaurant above 8% of weekly volume within 12 months.

---

### P3 — Protect and Grow the >$20 Segment

Orders above $20 represent 29% of volume but 60% of revenue. The margin economics make this segment disproportionately valuable to retain and expand.

**Action:** Identify the top 100 customers by lifetime spend and offer a loyalty tier with perks (free delivery above $25, priority courier matching). Recruit additional premium restaurants to expand the high-value order pool. Test order-value upsell prompts at checkout for orders in the $17–$20 range.

---

### P4 — Set Automated SLA Alerts at 45 Minutes

10.54% of orders exceed 60 minutes end-to-end — the threshold most correlated with low ratings and likely churn. A 45-minute alert provides a 15-minute window for proactive intervention.

**Action:** Trigger an automated customer notification at 45-minute total time with a revised ETA and a voucher offer. Measure impact on rating rates and repeat-order rates for affected orders vs. a control cohort. This is a high-ROI change with low implementation cost.